In [ ]:
import cupy as cp
import numpy as np
from time import perf_counter
from pyqcu import define
from pyqcu import io
from pyqcu import qcu
import cupyx.scipy.sparse.linalg as csla
from typing import Callable, Tuple, List

print('My rank is ', define.rank)

In [ ]:
params = np.array([0]*define._PARAMS_SIZE_, dtype=np.int32)
params[define._LAT_X_] = 32
params[define._LAT_Y_] = 32
params[define._LAT_Z_] = 32
params[define._LAT_T_] = 32
params[define._LAT_XYZT_] = 1048576
params[define._GRID_X_] = 1
params[define._GRID_Y_] = 1
params[define._GRID_Z_] = 1
params[define._GRID_T_] = 1
params[define._PARITY_] = 0
params[define._NODE_RANK_] = 0
params[define._NODE_SIZE_] = 1
params[define._DAGGER_] = 0
params[define._MAX_ITER_] = 1e4
params[define._DATA_TYPE_] = 0
params[define._SET_INDEX_] = 2
params[define._SET_PLAN_] = 0
argv = np.array([0.0]*define._ARGV_SIZE_, dtype=np.float32)
argv[define._MASS_] = 0.0
argv[define._TOL_] = 1e-9
print("Parameters:", params)
print("Arguments:", argv)

In [ ]:
gauge_filename = f"quda_wilson-dslash-gauge_-{params[define._LAT_X_]}-{params[define._LAT_Y_]}-{params  [define._LAT_Z_]}-{params[define._LAT_T_]}-{params[define._LAT_XYZT_]}-{params[define._GRID_X_]}-{params[define._GRID_Y_]}-{params[define._GRID_Z_]}-{params[define._GRID_T_]}-{params[define._PARITY_]}-{params[define._NODE_RANK_]}-{params[define._NODE_SIZE_]}-{params[define._DAGGER_]}-f.bin"
print("Gauge filename:", gauge_filename)
gauge = cp.fromfile(gauge_filename, dtype=cp.complex64,
                    count=params[define._LAT_XYZT_]*define._LAT_DCC_)
gauge = io.gauge2ccdptzyx(gauge, params)
print("Gauge:", gauge)
print("Gauge data:", gauge.data)
print("Gauge shape:", gauge.shape)

In [ ]:
set_ptrs = np.array(params, dtype=np.int64)
print("Set pointers:", set_ptrs)
print("Set pointers data:", set_ptrs.data)
qcu.applyInitQcu(set_ptrs, params, argv)

In [5]:
# give x, b, r, r_tilde, p, v, s, t
lat_t = params[define._LAT_T_]
lat_z = params[define._LAT_Z_]
lat_y = params[define._LAT_Y_]
lat_x = int(params[define._LAT_X_]/define._LAT_P_)
lat_d = define._LAT_D_
lat_s = define._LAT_S_
lat_p = define._LAT_P_
lat_c = define._LAT_C_
latt_shape = (lat_s, lat_c, lat_t, lat_z, lat_y, lat_x)
MAX_ITER = params[define._MAX_ITER_]
TOL = argv[define._TOL_]
N = params[define._LAT_XYZT_] * define._LAT_HALF_SC_
define._LAT_Ne_ = 10
max_eigen_value = 1.6
scale = 1e3

In [6]:
def _matvec(src):
    print("norm src", cp.linalg.norm(src))
    dest = cp.zeros(N, cp.complex64)
    qcu.applyWilsonCgDslashQcu(dest, src, gauge, set_ptrs, params)
    print("norm dest", cp.linalg.norm(dest))
    return dest


def __matvec(src, max_eigen_value=max_eigen_value, scale=scale):
    print("norm src", cp.linalg.norm(src))
    dest = cp.zeros(N, cp.complex64)
    qcu.applyWilsonCgDslashQcu(dest, src, gauge, set_ptrs, params)
    print("norm dest", cp.linalg.norm(dest))
    return max_eigen_value*src-scale*dest

In [7]:
# def power_iteration(matvec, v0, num_iter=params[define._MAX_ITER_], tol=argv[define._TOL_]):
#     v = v0
#     for i in range(num_iter):
#         Av = matvec(v)
#         v_new = Av / cp.linalg.norm(Av)
#         diff = cp.linalg.norm(v_new - v)
#         print(f"Iteration {i+1}: difference = {diff:.6e}")
#         if diff < tol:
#             print(
#                 f"Power Iteration converged after {i+1} iterations. Difference: {diff:.6e}")
#             break
#         v = v_new
#     eigenvalue = cp.dot(v.conj().T, matvec(v))
#     return eigenvalue, v
# def validate_power_iteration():
#     v0 = cp.random.randn(N).astype(cp.complex64)
#     eigenvalue, eigenvector = power_iteration(_matvec, v0)
#     print("Dominant eigenvalue:", eigenvalue)
#     print("Corresponding eigenvector:", eigenvector[:10])
#     Ax = _matvec(eigenvector)
#     eigenvalue_check = cp.dot(eigenvector.conj().T, Ax)
#     print(f"Eigenvalue verification (v^T A v): {eigenvalue_check}")
#     if cp.isclose(eigenvalue, eigenvalue_check):
#         print("Validation passed: Eigenvalue calculation is correct.")
#     else:
#         print("Validation failed: Eigenvalue calculation is incorrect.")
#     print("Eigenvector norm:", cp.linalg.norm(eigenvector))
#     return eigenvalue, eigenvector
# eigenvalue, eigenvector = validate_power_iteration()
# print("Diff:", cp.linalg.norm(_matvec(eigenvector) -
#       eigenvalue*eigenvector) / cp.linalg.norm(eigenvector))

In [8]:
# print("Eigenvalue:", eigenvalue)

In [9]:
# # bistabcg
# def dot(a, b):
#     cp.cuda.runtime.deviceSynchronize()
#     return cp.inner(a.flatten().conjugate(), b.flatten())


# def diff(a, b):
#     cp.cuda.runtime.deviceSynchronize()
#     return cp.linalg.norm(a - b) / cp.linalg.norm(b)


# def run(eigen_value):
#     t0 = perf_counter()
#     x = cp.random.rand(N).astype(cp.complex64)
#     print("shape:", x.shape)
#     b = cp.zeros(N, cp.complex64)  # must be zero
#     r = cp.zeros(N, cp.complex64)
#     r_tilde = cp.zeros(N, cp.complex64)
#     p = cp.zeros(N, cp.complex64)
#     s = cp.zeros(N, cp.complex64)
#     v = cp.zeros(N, cp.complex64)
#     t = cp.zeros(N, cp.complex64)
#     r = b - _matvec(x, eigen_value)
#     r_tilde = r
#     r_norm2 = 0
#     rho_prev = 1
#     rho = 0
#     alpha = 1
#     omega = 1
#     beta = 0
#     for i in range(MAX_ITER):
#         # print("## rho:", rho)
#         rho = dot(r_tilde, r)
#         # print("## beta:", beta)
#         beta = (rho / rho_prev) * (alpha / omega)
#         p = r + (p - v * omega) * beta
#         # v = A * p
#         v = _matvec(p, eigen_value)
#         # print("## alpha:", alpha)
#         alpha = rho / dot(r_tilde, v)
#         s = r - v * alpha
#         # t = A * s
#         t = _matvec(s, eigen_value)
#         # print("## omega:", omega)
#         omega = dot(t, s) / dot(t, t)
#         x = x + p * alpha + s * omega
#         r = s - t * omega
#         r_norm2 = cp.linalg.norm(r)
#         print("##{}# r_norm2:{}".format(i, r_norm2))
#         _diff = diff(_matvec(x, 0.0), eigen_value*x)
#         print("diff:", _diff)
#         # break
#         # if (r_norm2 < TOL or i == MAX_ITER - 1):
#         #     print("### turns:", i)
#         #     break
#         if (_diff < TOL or i == MAX_ITER - 1):
#             print("### turns:", i)
#             break
#         rho_prev = rho
#     # print(" diff:", diff(_matvec(x, eigen_value), b))
#     # print("x:", x)
#     t1 = perf_counter()
#     print("Time:", t1-t0)
#     return x, _diff


# eigen_value = eigenvalue
# eigen_vector, _diff = run(eigen_value)
# print("eigen_vector:", eigen_vector)
# print("eigen_vector shape:", eigen_vector.shape)
# print("diff:", _diff)

In [10]:
# print("Difference:", cp.linalg.norm(eigenvector-eigen_vector)/cp.linalg.norm(eigenvector))

In [11]:
# import cupyx.scipy.sparse.linalg as csla
# define._LAT_Ne_ = 24
# A = csla.LinearOperator((N,
#                         N), matvec=_matvec, dtype=cp.complex64)
# evals, evecs = csla.eigsh(a=A, k=define._LAT_Ne_, which="SA",
#                           return_eigenvectors=True)
# print("evals:", evals)
# print("evecs:", evecs)
# cp.linalg.norm(_matvec(evecs[:, 0])-evals[0] *
#                evecs[:, 0])/cp.linalg.norm(evals[0]*evecs[:, 0])

In [ ]:
def shifted_power_iteration(
    matvec,
    n=N,
    num_eigenvalues=define._LAT_Ne_,
    max_iter=params[define._MAX_ITER_],
    tol=argv[define._TOL_]
):
    eigenvalues = []
    eigenvectors = []
    # Buffer for intermediate results
    v = cp.ndarray((n,), dtype=np.complex64)
    w = cp.ndarray((n,), dtype=np.complex64)
    # Store cumulative shift
    total_shift = 0.0

    for k in range(num_eigenvalues):
        # Initialize random vector
        v[:] = cp.random.rand(n) + 1j * cp.random.rand(n)
        # Orthogonalize against previously found eigenvectors
        for j in range(k):
            projection = cp.dot(cp.conj(eigenvectors[j]), v)
            v -= projection * eigenvectors[j]
        # Normalize
        v /= cp.linalg.norm(v)

        # Power iteration
        eigenvalue = None
        for iter_count in range(max_iter):
            # Matrix-vector multiplication
            w = matvec(v)
            # Apply shift
            w += total_shift * v
            # Calculate Rayleigh quotient
            new_eigenvalue = cp.dot(cp.conj(v), w)
            # Normalize
            norm = cp.linalg.norm(w)
            if norm < tol:  # Avoid zero vector
                v[:] = cp.random.rand(n) + 1j * cp.random.rand(n)
                continue
            w /= norm

            # Check convergence
            if eigenvalue is not None:
                diff = abs(new_eigenvalue - eigenvalue)
                print(f"Iteration {iter_count} for eigenvalue {k}: {diff:.8f}")
                if diff < tol:
                    break
            eigenvalue = new_eigenvalue
            v[:] = w

        # Store results and subtract shift
        if eigenvalue is not None:
            eigenvalue -= total_shift
            eigenvalues.append(eigenvalue)
            eigenvectors.append(v.copy())
            # Update shift for next eigenvalue
            total_shift = eigenvalue.real

    return eigenvalues, eigenvectors


# Calculate eigenvalues and eigenvectors
eigenvalues, eigenvectors = shifted_power_iteration(__matvec)

# Verify results
print("Computed eigenvalues:")
for i, ev in enumerate(eigenvalues):
    print(f"λ_{i} = {ev:.8f}")
    # Verify eigenvector
    v = eigenvectors[i]
    w = cp.zeros_like(v)
    w = __matvec(v)
    error = cp.linalg.norm(w - ev * v) / cp.linalg.norm(w)
    print(f"Relative error: {error:.2e}")

In [ ]:
eigenvalues

In [ ]:
eigenvectors[0]

In [ ]:
eigenvectors[1]

In [ ]:
eigenvectors[2]

In [ ]:
eigenvectors[3]

In [ ]:
eigen_index = 8
print("Diff:", cp.linalg.norm(__matvec(eigenvectors[eigen_index]) -
      eigenvalues[eigen_index]*eigenvectors[eigen_index]) / cp.linalg.norm(eigenvectors[eigen_index]))

In [ ]:
for i, ev in enumerate(eigenvalues):
    print(f"λ_{i} = {ev:.8f}")
    # 验证本征向量
    v = eigenvectors[i]*1e6
    print(f"v_{i} = {v}")
    w = cp.zeros_like(v)
    w = __matvec(v)
    print(f"w_{i} = {w}")
    print(f"ev_{i} * v_{i} = {ev*v}")
    error = cp.linalg.norm(w - ev * v) / cp.linalg.norm(w)
    print(f"相对误差: {error:.2e}")
    j = i+1
    if i == len(eigenvalues)-1:
        j = 0
    eigen_diff = cp.linalg.norm(
        eigenvectors[i]-eigenvectors[j])/cp.linalg.norm(eigenvectors[i])
    print(f"本征向量{i}与本征向量{j}的相对误差:{eigen_diff:.2e}")

In [20]:
_eigenvalues = (max_eigen_value -
                cp.array(eigenvalues, dtype=cp.complex64))/scale
_eigenvectors = cp.array(eigenvectors, dtype=cp.complex64)

In [ ]:
print("_eigenvalues:", _eigenvalues)

In [ ]:
print("_eigenvectors.shape:", _eigenvectors.shape)
print(_eigenvectors)

In [ ]:

for i, ev in enumerate(_eigenvalues):
    print(f"λ_{i} = {ev:.8f}")
    # 验证本征向量
    v = _eigenvectors[i]
    print(f"v_{i} = {v}")
    w = cp.zeros_like(v)
    w = _matvec(v)
    print(f"w_{i} = {w}")
    print(f"ev_{i} * v_{i} = {ev*v}")
    error = cp.linalg.norm(w - ev * v) / cp.linalg.norm(w)
    print(f"相对误差: {error:.2e}")
    j = i+1
    if i == len(_eigenvalues)-1:
        j = 0
    eigen_diff = cp.linalg.norm(
        _eigenvectors[i]-_eigenvectors[j])/cp.linalg.norm(_eigenvectors[i])
    print(f"本征向量{i}与本征向量{j}的相对误差:{eigen_diff:.2e}")

In [24]:
# import numpy as np

# def min_eigenvalue_power_iteration(A, num_iterations=100, tolerance=1e-10):
#     """
#     使用幂迭代法计算矩阵的最小本征值

#     参数:
#         A: numpy数组，输入矩阵（方阵）
#         num_iterations: 最大迭代次数
#         tolerance: 收敛容差

#     返回:
#         min_eigenvalue: 最小本征值的估计值
#         min_eigenvector: 对应的本征向量
#     """
#     # 计算矩阵的最大绝对本征值（用于位移变换）
#     n = A.shape[0]
#     v = np.random.rand(n)
#     max_eigenvalue = power_iteration(A, num_iterations, tolerance)[0]

#     # 进行位移变换：B = max_eigenvalue * I - A
#     # 这样B的最大本征值对应A的最小本征值
#     I = np.eye(n)
#     B = max_eigenvalue * I - A

#     # 对变换后的矩阵使用标准幂迭移法
#     v = np.random.rand(n)
#     v = v / np.linalg.norm(v)

#     for i in range(num_iterations):
#         v_old = v.copy()

#         # 幂迭代步骤
#         v = B @ v
#         v = v / np.linalg.norm(v)

#         # 检查收敛性
#         if np.allclose(v, v_old, rtol=tolerance):
#             break

#     # 计算最小本征值
#     min_eigenvalue = max_eigenvalue - (v.T @ B @ v) / (v.T @ v)

#     return min_eigenvalue, v

# def power_iteration(A, num_iterations=100, tolerance=1e-10):
#     """
#     标准幂迭代法计算最大本征值
#     """
#     n = A.shape[0]
#     v = np.random.rand(n)
#     v = v / np.linalg.norm(v)

#     for i in range(num_iterations):
#         v_old = v.copy()
#         v = A @ v
#         v = v / np.linalg.norm(v)

#         if np.allclose(v, v_old, rtol=tolerance):
#             break

#     eigenvalue = (v.T @ A @ v) / (v.T @ v)
#     return eigenvalue, v

# # 测试代码
# if __name__ == "__main__":
#     # 创建一个对称矩阵进行测试
#     A = np.array([[4, -1, 0],
#                   [-1, 4, -1],
#                   [0, -1, 4]])

#     min_eval, min_evec = min_eigenvalue_power_iteration(A)
#     print(f"最小本征值: {min_eval:.6f}")
#     print(f"对应的本征向量: {min_evec}")

#     # 验证结果
#     true_eigenvalues = np.linalg.eigvals(A)
#     print(f"numpy计算的所有本征值: {true_eigenvalues}")
#     print(f"实际最小本征值: {min(true_eigenvalues.real):.6f}")

In [25]:
# qcu.applyEndQcu(set_ptrs, params)